# Baseline: TF-IDF + Logistic Regression

This notebook demonstrates a baseline experiment using TF-IDF embeddings and Logistic Regression classifier on the IMDB sentiment dataset.

## Setup

If running on Google Colab, uncomment and run the cell below to mount Google Drive or download the dataset.

In [ ]:
# Uncomment for Google Colab
# !pip install -q gensim transformers torch scikit-learn

# Option 1: Mount Google Drive (if dataset is stored there)
# from google.colab import drive
# drive.mount('/content/drive')
# !tar -xzf /content/drive/MyDrive/path/to/aclImdb_v1.tar.gz -C /content/

# Option 2: Download dataset directly
# !wget -P /content/ http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# !tar -xzf /content/aclImdb_v1.tar.gz -C /content/

# Clone repository (if needed)
# !git clone https://github.com/rylex27-z/Embedding_models_with_Classification.git
# %cd Embedding_models_with_Classification

## Imports

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from imdb_benchmark.data_loader import load_imdb_data, get_dataset_info
from imdb_benchmark.embeddings.registry import create_embedding
from imdb_benchmark.models.registry import create_model
from imdb_benchmark.tuning import CVEvaluator, aggregate_cv_results

import pandas as pd
import logging

logging.basicConfig(level=logging.INFO)

## Load Data

Update `DATA_PATH` to point to your extracted `aclImdb` directory.

In [ ]:
# Update this path!
DATA_PATH = "/content/aclImdb"  # For Colab
# DATA_PATH = "/path/to/aclImdb"  # For local

# Load data
X_train, y_train, X_test, y_test = load_imdb_data(DATA_PATH)

# Show dataset info
info = get_dataset_info(DATA_PATH)
print("Dataset Information:")
for key, value in info.items():
    print(f"  {key}: {value}")

## Run Baseline Experiment

We'll use:
- **Embedding**: TF-IDF with unigrams and bigrams
- **Model**: Logistic Regression
- **CV Protocol**: 5-fold × 4 seeds = 20 evaluations

In [ ]:
# Create embedding and model factories
def embedding_fn():
    return create_embedding("tfidf_bigram")

def model_fn():
    return create_model("logreg", random_state=42)

# Create CV evaluator
evaluator = CVEvaluator(
    n_folds=5,
    seeds=[42, 123, 456, 789],
    tune_hyperparams=False,
)

# Run cross-validation
results_df = evaluator.evaluate(
    embedding_fn=embedding_fn,
    model_fn=model_fn,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    embedding_name="tfidf_bigram",
    model_name="logreg",
)

## View Results

In [ ]:
# Display detailed results
print("\nDetailed Results:")
print(results_df.head(10))

# Aggregate results
summary = aggregate_cv_results(results_df)
print("\nAggregated Results:")
print(summary)

## Save Results

In [ ]:
# Create output directory
output_dir = Path("reports/results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save detailed results
results_file = output_dir / "results_long.csv"
results_df.to_csv(results_file, index=False)
print(f"Results saved to {results_file}")

# Save summary
summary_file = output_dir / "results_summary.csv"
summary.to_csv(summary_file, index=False)
print(f"Summary saved to {summary_file}")

## Commit Results to GitHub

If running on Colab, you can download the results files and commit them manually, or use the following cells to commit directly from Colab (requires authentication).

In [ ]:
# Download results files (for Colab)
# from google.colab import files
# files.download(str(results_file))
# files.download(str(summary_file))

In [ ]:
# Commit to GitHub (requires git configuration)
# !git config --global user.email "your-email@example.com"
# !git config --global user.name "Your Name"
# !git add reports/results/*.csv
# !git commit -m "Add baseline TF-IDF + LogReg results"
# !git push